# Day 1 — Setup check

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Goal for today is only to prove the plumbing works:
environment reports correctly, the model loads, generation runs, and the KV cache is reachable.
Nothing project-specific is built yet.

In [ ]:
!pip install -q transformers accelerate
import torch, platform, sys
print("python  :", sys.version.split()[0])
print("torch   :", torch.__version__)
print("cuda    :", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("gpu     :", p.name, f"({p.total_memory/1024**3:.1f} GB)")
    print("bf16 ok :", torch.cuda.is_bf16_supported(), "<- False on T4, so use fp16")

## Load the model

Two choices worth understanding:

- **fp16, not bf16** — the T4 is a Turing card and has no bf16 support.
- **`attn_implementation="eager"`** — the faster `sdpa` kernel does not return attention
  weights, and later phases need them to score tokens.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # small on purpose: fast to iterate

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = "left"   # decoder-only models must pad left

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    attn_implementation="eager",
).to("cuda").eval()
print(f"loaded in {time.time()-t0:.1f}s")

## Model shape

`num_key_value_heads < num_attention_heads` means the model uses **GQA** — several query
heads share one key/value head. This is edge case #8 in the plan: when scoring tokens you
must combine the scores of a group *before* selecting, or selection silently degrades.

In [ ]:
c = model.config
n_kv = getattr(c, "num_key_value_heads", c.num_attention_heads)
head_dim = getattr(c, "head_dim", c.hidden_size // c.num_attention_heads)

print("layers          :", c.num_hidden_layers)
print("query heads     :", c.num_attention_heads)
print("key/value heads :", n_kv)
print("head_dim        :", head_dim)
print("GQA group size  :", c.num_attention_heads // n_kv)

## Generate

`do_sample=False` (greedy) everywhere in this project — sampling makes runs
unreproducible and breaks the identity test on Day 4.

In [ ]:
prompt = "Explain what a KV cache is in a transformer, in two sentences."
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(text, return_tensors="pt").to("cuda")
n_prompt = inputs["input_ids"].shape[1]

t0 = time.time()
with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=48, do_sample=False,
        return_dict_in_generate=True, use_cache=True,
    )
dt = time.time() - t0
n_new = out.sequences.shape[1] - n_prompt

print(tokenizer.decode(out.sequences[0][n_prompt:], skip_special_tokens=True).strip())
print(f"\n{n_new} tokens in {dt:.1f}s ({n_new/dt:.1f} tok/s)")

## Look at the KV cache

This is the object the entire project compresses. Confirm it is reachable and shaped as expected:
`(batch, kv_heads, tokens, head_dim)` per layer.

In [ ]:
cache = out.past_key_values
key_0 = cache[0][0]

print("layers in cache :", len(cache))
print("layer 0 key     :", tuple(key_0.shape), " (batch, kv_heads, tokens, head_dim)")

assert len(cache) == c.num_hidden_layers
assert key_0.shape[1] == n_kv
assert key_0.shape[2] == n_prompt + n_new - 1
print("\nDAY 1 PASSED")

---

**Day 2 next:** compute KV cache size by hand from the config and check it against the real
tensor sizes. Formula:

```
bytes = 2 * num_layers * num_key_value_heads * head_dim * num_tokens * dtype_bytes
```

(the leading 2 is for keys *and* values)